In [ ]:
from varphot import nbui as ui

ui.show(ui.header("05", "Archivierung und Projektdokumentation",
    "strukturiertes Archiv im Datenordner mit `zeitreihe.csv`, AAVSO-Datei, Plots, `bericht.md` (Nacht-Bericht), `index.md` (Ziel-Übersicht), Excel und einer kumulativen PDF-Zusammenfassung aller bisherigen Nächte je Ziel.",
    wann="nach NB03 (AAVSO-Datei erstellt)",
    flow=[("Schritt 1 — Formular starten", "Die Zellen von oben nach unten ausführen (▶ oder Shift+Enter), bis das **Formular mit den Eingabefeldern** erscheint"),
          ("Schritt 2 — Prüfen", "Meist Defaults übernehmen; der **Hinweiskasten im Formular** zeigt: **grün** = ok · **orange** = prüfen · **rot** = Fehler"),
          ("Schritt 3 — Automatisch", "Weiter mit dem Abschnitt **„Ab hier automatisch“**: bei der nächsten Code-Zelle den Pfeil (▶) öffnen → **„Diese Zelle und alle folgenden“** ausführen")],
    blocks=[ui.kv([("Archivordner", "wo alles abgelegt wird (Vorgabe: `<Datenordner>/archiv/`)"), ("Bildauswahl", "welche Einzelbilder kopiert werden"), ("Excel", "Zeitreihe als `.xlsx`"),
                   ("PDF", "automatisch `<ziel>_beobachtungen.pdf`, kumulative Zusammenfassung aller Nächte (braucht Pandoc und XeLaTeX, siehe NB00 Schritt 1.1)")], 130)]))

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("05", None, "Formular", blocks=[ui.tasks(["Die Code-Zelle direkt unter dieser Karte ausführen (▶): das Formular erscheint", "Archivordner, Bildauswahl und Excel-Option im Formular festlegen", "Weiter mit dem Abschnitt „Ab hier automatisch“: bei der nächsten Code-Zelle den Pfeil (▶) öffnen und „Diese Zelle und alle folgenden“ ausführen"])]))

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from varphot.hints import bind
from varphot.observer import data_dir as _data_dir
from varphot.archive import archive_dir_default, load_archive_defaults

_dd = _data_dir()
_def = load_archive_defaults(_dd if _dd.is_dir() else None)

w_archivdir = widgets.Text(
    description="Archivordner",
    value=_def.get("archive_root", str(archive_dir_default(_dd if _dd.is_dir() else None))),
    layout=widgets.Layout(width="560px"),
)
w_images = widgets.RadioButtons(
    options=[
        ("Kein Einzelbild", "none"),
        ("Erstes Bild", "first"),
        ("Schärfstes Bild (best)", "best"),
        ("Alle Bilder (kann sehr groß sein!)", "all"),
    ],
    value=_def.get("images", "best"),
    description="Einzelbilder",
    layout=widgets.Layout(width="420px"),
)
w_messtabelle = widgets.Checkbox(
    value=_def.get("copy_messtabelle", False),
    description="messtabelle.csv kopieren (groß, enthält alle Einzelmessungen)",
)
w_excel = widgets.Checkbox(
    value=_def.get("excel", True),
    description="Excel-Export (zeitreihe.xlsx)",
)
w_save_defaults = widgets.Checkbox(
    value=True,
    description="Diese Einstellungen als Vorgabe speichern",
)

bind(w_archivdir, "nb05_archivdir")
bind(w_images, "nb05_images")

display(widgets.VBox([
    w_archivdir,
    w_images,
    widgets.HBox([w_messtabelle, w_excel]),
    w_save_defaults,
]))

## Ab hier automatisch

Bei der **nächsten Code-Zelle** den Pfeil (▶) öffnen → **„Diese Zelle und alle folgenden“** ausführen.

In [ ]:
from pathlib import Path
from varphot import Session
from varphot.archive import archive_session, save_archive_defaults
from varphot.observer import data_dir as _data_dir

SESS = Session.current()
print(f"Sitzung: {SESS.target} | {SESS.tag} | {SESS.aavso_filter}")
print(f"Ergebnisse: {SESS.results_dir()}")

ARCHIVE_ROOT = Path(w_archivdir.value.strip()).expanduser()
IMAGES       = w_images.value
MESSTABELLE  = bool(w_messtabelle.value)
EXCEL        = bool(w_excel.value)
if w_save_defaults.value:
    _dd = _data_dir()
    save_archive_defaults({
        "archive_root": str(ARCHIVE_ROOT),
        "images": IMAGES,
        "copy_messtabelle": MESSTABELLE,
        "excel": EXCEL,
    }, _dd if _dd.is_dir() else None)
    print("✓ Vorgaben gespeichert")

In [ ]:
DEST = archive_session(
    SESS,
    archive_root=ARCHIVE_ROOT,
    images=IMAGES,
    copy_messtabelle=MESSTABELLE,
    excel=EXCEL,
)

In [ ]:
# Bericht im Notebook anzeigen
from IPython.display import Markdown
bericht = (DEST / "bericht.md").read_text(encoding="utf-8")
display(Markdown(bericht))

In [ ]:
# Ziel-Index anzeigen
from varphot.archive import _safe_name
index_path = ARCHIVE_ROOT / _safe_name(SESS.target or "unbekannt") / "index.md"
if index_path.exists():
    display(Markdown(index_path.read_text(encoding="utf-8")))
else:
    print("(Noch kein Index — wird beim nächsten Archivlauf angelegt)")